# 03 — Model Research

Walk-forward training across 8 folds with LightGBM, RandomForest, XGBoost.

In [ ]:
import sys, yaml
import pandas as pd
from pathlib import Path

sys.path.insert(0, str(Path('..')))
from src.features import generate_features
from src.models import get_models
from src.walk_forward import run_walk_forward

with open('../configs/strategy.yaml') as f:
    cfg = yaml.safe_load(f)

HORIZON_BARS = cfg['horizon_bars']
FEATURE_COLS = cfg['features']
N_FOLDS      = cfg['walk_forward']['n_folds']

df_raw = pd.read_csv('../data/ETHUSDT.csv', parse_dates=['timestamp'], index_col='timestamp')
df_features, target = generate_features(df_raw, feature_cols=FEATURE_COLS, HORIZON_BARS=HORIZON_BARS)

### Walk-Forward Validation: 20% Train, 10% Valid, 10% Step

8 folds, each fold trains on the 20% window immediately before the 10% validation window.

In [ ]:
models = get_models()
X_total, result = run_walk_forward(
    df_features, target, models,
    HORIZON_BARS=HORIZON_BARS,
    n_folds=N_FOLDS
)

In [ ]:
# Persist results for 04_results notebook
import pickle
Path('../data').mkdir(exist_ok=True)
with open('../data/wf_results.pkl', 'wb') as f:
    pickle.dump({'X_total': X_total, 'result': dict(result)}, f)
print('Saved.')